# Transformer-based NER

In [2]:
! pip install torch --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118


In [3]:
! pip install -U spacy[transformers]

In [2]:
# ! python -m spacy init config config_tr.cfg --lang en --pipeline transformer,ner --optimize efficiency --force -G

/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
ℹ Generated config template specific for your use case
- Language: en
- Pipeline: ner
- Optimize for: efficiency
- Hardware: GPU
- Transformer: roberta-base
✔ Auto-filled config with all values
✔ Saved config
config_tr.cfg
You can now add your data and train your pipeline:
python -m spacy train config_tr.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [4]:
! python -m spacy train config.cfg --output ./transformer --paths.train ./train_data.spacy --paths.dev ./val_data.spacy

✔ Created output directory: transformer
ℹ Saving to output directory: transformer
ℹ Using CPU

=========================== Initializing pipeline ===========================
/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ---

In [1]:
import spacy
from spacy.training import Corpus

nlp_ner = spacy.load("./transformer/model-best")
print(nlp_ner.pipe_names)

# Load test data (adjust path to your test data)
test_data_path = "/home/yp6443/research/nlp/taxigen/test_data.spacy"

# If your test data is already in .spacy format:
corpus = Corpus(test_data_path)
test_examples = corpus(nlp_ner)

# Evaluate the model
scores = nlp_ner.evaluate(test_examples)

# Print NER metrics
print("=== Tok2Vec w/o Heuristics ===")
print("=== NER Evaluation Metrics ===")
print(f"Precision: {scores['ents_p']:.3f}")    # Use ['ents_p']
print(f"Recall:    {scores['ents_r']:.3f}")    # Use ['ents_r']
print(f"F1-Score:  {scores['ents_f']:.3f}")    # Use ['ents_f']

/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


['tok2vec', 'ner']
=== Tok2Vec w/o Heuristics ===
=== NER Evaluation Metrics ===
Precision: 0.859
Recall:    0.559
F1-Score:  0.677


In [2]:
import glob
test_file = glob.glob('/home/yp6443/research/nlp/voice_data/test_file/*.txt')
file_idx = 1

with open(test_file[file_idx], 'r') as file:
    for line in file:
        doc = nlp_ner(line.strip())
        spacy.displacy.render(doc, style="ent", jupyter=True)

/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/spacy/displacy/__init__.py:213: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)


In [3]:
nlp_ner = spacy.load("./transformer/model-best")

ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": True}  
)
ruler.from_disk("entity_rulers.jsonl")

with open(test_file[file_idx], 'r') as file:
    for line in file:
        doc = nlp_ner(line.strip())
        spacy.displacy.render(doc, style="ent", jupyter=True)

/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/spacy/displacy/__init__.py:213: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)


In [4]:
nlp_ner = spacy.load("./transformer/model-best")

ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": False}  
)
ruler.from_disk("entity_rulers.jsonl")
print(nlp_ner.pipe_names)

# Load test data (adjust path to your test data)
test_data_path = "/home/yp6443/research/nlp/taxigen/test_data.spacy"

# If your test data is already in .spacy format:
corpus = Corpus(test_data_path)
test_examples = corpus(nlp_ner)

# Evaluate the model
scores = nlp_ner.evaluate(test_examples)

# Print NER metrics
print("=== Tok2Vec w/ Heuristics ===")
print("=== NER Evaluation Metrics ===")
print(f"Precision: {scores['ents_p']:.3f}")    
print(f"Recall:    {scores['ents_r']:.3f}")
print(f"F1-Score:  {scores['ents_f']:.3f}")    

['tok2vec', 'ner', 'entity_ruler']
=== Tok2Vec w/ Heuristics ===
=== NER Evaluation Metrics ===
Precision: 0.805
Recall:    0.678
F1-Score:  0.736


# Train with entity rule spans

In [9]:
! python -m spacy train config_tr.cfg \
    --output ./transformer_rules \
        --paths.train ./augmented_train_data.spacy \
            --paths.dev ./val_data.spacy \
                --gpu-id 0


/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
ℹ Saving to output directory: transformer_rules
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/yp6443/anaconda3/envs/sedona/lib/python3.10/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/

In [10]:
from spacy.training import Corpus

nlp_ner = spacy.load("./transformer_rules/model-best")

# Load test data (adjust path to your test data)
test_data_path = "/home/yp6443/research/nlp/taxigen/test_data.spacy"

# If your test data is already in .spacy format:
corpus = Corpus(test_data_path)
test_examples = corpus(nlp_ner)

# Evaluate the model
scores = nlp_ner.evaluate(test_examples)

# Print NER metrics
print("=== Tok2Vec trained w/ Heuristics w/o Heuristics override ===")
print("=== NER Evaluation Metrics ===")
print(f"Precision: {scores['ents_p']:.3f}")    
print(f"Recall:    {scores['ents_r']:.3f}")
print(f"F1-Score:  {scores['ents_f']:.3f}")    

NameError: name 'spacy' is not defined

In [ ]:
nlp_ner = spacy.load("./transformer_rules/model-best")

ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": False}  
)
ruler.from_disk("entity_rulers.jsonl")
print(nlp_ner.pipe_names)

# Load test data (adjust path to your test data)
test_data_path = "/home/yp6443/research/nlp/taxigen/test_data.spacy"

# If your test data is already in .spacy format:
corpus = Corpus(test_data_path)
test_examples = corpus(nlp_ner)

# Evaluate the model
scores = nlp_ner.evaluate(test_examples)

# Print NER metrics
print("=== Tok2Vec trained w/ Heuristics w/ Heuristics override ===")
print("=== NER Evaluation Metrics ===")
print(f"Precision: {scores['ents_p']:.3f}")    
print(f"Recall:    {scores['ents_r']:.3f}")
print(f"F1-Score:  {scores['ents_f']:.3f}")    